In [1]:
import sys
sys.path.append('..')

In [2]:
from utils.prompts import render
from utils.router import pick_model
from utils.llm_client import LLMClient
from utils.config_loader import reload_config
from pathlib import Path
reload_config()

In [3]:
incident_path = Path("../data/Incidents.txt")

with open(incident_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

incidents = []

# skip header
for line in lines[1:]:

    line = line.strip()

    if not line:
        continue

    parts = [part.strip() for part in line.split("|")]

    incident = {
        "id": parts[0],
        "time": parts[1],
        "area": parts[2],
        "people": parts[3],
        "ages": parts[4],
        "main_need": parts[5],
        "message": parts[6].strip('"')
    }

    incidents.append(incident)

print(incidents)

[{'id': '1', 'time': '08:00 AM', 'area': 'Gampaha', 'people': '4', 'ages': '20-40', 'main_need': 'Water', 'message': 'Thirsty but safe on roof. Water level stable.'}, {'id': '2', 'time': '08:15 AM', 'area': 'Ja-Ela', 'people': '1', 'ages': '75', 'main_need': 'Insulin', 'message': 'Diabetic, missed dose yesterday. Feeling faint.'}, {'id': '3', 'time': '08:20 AM', 'area': 'Ragama', 'people': '2', 'ages': '10, 35', 'main_need': 'Rescue', 'message': 'Water approaching neck level. Child is crying.'}]


In [4]:
def score_incident(incident, llm):

    prompt_text, _ = render(
        "cot_reasoning.v1",

        role="crisis incident priority evaluator",

        query=incident,

        instruction="""
Calculate the priority score in stages:

1. Start with a base score of 5.
2. Check the ages.
   - Add 2 points if any victim is older than 60 or younger than 5.
3. Check the main need.
   - Add 3 points if the main need is Rescue.
4. Check whether medicine is required.
   - Add 1 point if the incident requires medicine.
5. Add all applicable points.
6. Return the final priority score.
""",

        constraints="""
Use only the scoring rules provided above.
Do not invent additional scoring rules.
Do not add points more than once for the same rule.
""",

        format="""
Base Score: [score]
Age Bonus: [score]
Rescue Bonus: [score]
Medicine Bonus: [score]
Final Priority Score: [score]
Reason: [brief explanation]
"""
    )

    response = llm.chat(
        [{"role": "user", "content": prompt_text}],
        temperature=0.0,
        task_type="reasoning"
    )

    return response["text"].strip()

In [5]:
model = pick_model(
    provider="groq",
    technique="cot_reasoning"
)

print(model)

llm = LLMClient(
    "groq",
    model
)

openai/gpt-oss-120b


In [6]:
import re

scored_incidents = []

for incident in incidents:

    result = score_incident(
        incident,
        llm
    )

    clean_result = result.replace("**", "")

    match = re.search(
        r"Final Priority Score:\s*(\d+)",
        clean_result
    )

    score = int(match.group(1))

    scored_incidents.append({
        "id": incident["id"],
        "area": incident["area"],
        "main_need": incident["main_need"],
        "score": score,
        "cot_result": result
    })

    print("=" * 70)
    print(incident)
    print(result)

{'id': '1', 'time': '08:00 AM', 'area': 'Gampaha', 'people': '4', 'ages': '20-40', 'main_need': 'Water', 'message': 'Thirsty but safe on roof. Water level stable.'}
**Base Score:** 5  
**Age Bonus:** 0  
**Rescue Bonus:** 0  
**Medicine Bonus:** 0  
**Final Priority Score:** 5  

**Reason:** The victims are aged 20‑40, which does not meet the age‑based bonus criteria. The main need is water, not rescue, and there is no indication that medicine is required. Therefore, only the base score applies.
{'id': '2', 'time': '08:15 AM', 'area': 'Ja-Ela', 'people': '1', 'ages': '75', 'main_need': 'Insulin', 'message': 'Diabetic, missed dose yesterday. Feeling faint.'}
Base Score: 5  
Age Bonus: 2  
Rescue Bonus: 0  
Medicine Bonus: 1  
Final Priority Score: 8  
Reason: The victim is 75 years old (adds age bonus), the main need is insulin (medicine) not rescue, and medication is required, giving the additional medicine bonus.
{'id': '3', 'time': '08:20 AM', 'area': 'Ragama', 'people': '2', 'ages':

In [7]:
print(scored_incidents)

[{'id': '1', 'area': 'Gampaha', 'main_need': 'Water', 'score': 5, 'cot_result': '**Base Score:** 5  \n**Age Bonus:** 0  \n**Rescue Bonus:** 0  \n**Medicine Bonus:** 0  \n**Final Priority Score:** 5  \n\n**Reason:** The victims are aged 20‑40, which does not meet the age‑based bonus criteria. The main need is water, not rescue, and there is no indication that medicine is required. Therefore, only the base score applies.'}, {'id': '2', 'area': 'Ja-Ela', 'main_need': 'Insulin', 'score': 8, 'cot_result': 'Base Score: 5  \nAge Bonus: 2  \nRescue Bonus: 0  \nMedicine Bonus: 1  \nFinal Priority Score: 8  \nReason: The victim is 75 years old (adds age bonus), the main need is insulin (medicine) not rescue, and medication is required, giving the additional medicine bonus.'}, {'id': '3', 'area': 'Ragama', 'main_need': 'Rescue', 'score': 8, 'cot_result': 'Base Score: 5  \nAge Bonus: 0  \nRescue Bonus: 3  \nMedicine Bonus: 0  \nFinal Priority Score: 8  \nReason: The incident has a base score of 5,

In [8]:
def plan_rescue_strategy(scored_incidents, llm):

    prompt_text, _ = render(
        "tot_reasoning.v1",

        role="crisis logistics commander",

        query=scored_incidents,

        instruction="""
Explore exactly three strategy branches using the incident information
exactly as provided in the query.

Do not change or remap any Incident ID, Area, Main Need, or Priority Score.

Branch 1 - Highest Score First (Greedy):
1. Identify the incident or incidents with the highest priority score.
2. If there is a tie, use only the explicitly known travel time
   as the tie-breaker.
3. The incident at Ragama has 0 minutes initial travel because
   the rescue boat starts at Ragama.
4. Create the service order.
5. Evaluate the advantages and disadvantages.

Branch 2 - Closest First (Speed):
1. Start from Ragama.
2. The Ragama incident is at the starting location, so it is
   the closest incident with 0 minutes initial travel.
3. After serving Ragama, select the next closest incident using
   only explicitly provided travel times.
4. Continue toward the remaining incident.
5. Evaluate the route.

Branch 3 - Furthest First (Logistics):
1. Identify the furthest incident reachable through the provided
   forward travel legs.
2. Prioritize that incident first.
3. Passing through another location does not automatically mean
   that its incident has been served.
4. If completing the remaining route requires an unknown travel
   time, state "Unknown" instead of assuming a value.
5. Evaluate the consequences of delaying the closer incidents.

After evaluating all three branches:
- compare the priority scores served
- compare only known travel times
- compare how quickly high-priority victims are reached
- identify any unknown travel times
- select the route that serves the highest-priority incidents
  as early as possible while minimizing known travel time
""",

        constraints="""
There is only ONE rescue boat.

Starting location:
Ragama

Known travel times:
- Ragama -> Ja-Ela = 10 minutes
- Ja-Ela -> Gampaha = 40 minutes

The Ragama incident is at the boat's starting location,
so the initial travel time to that incident is 0 minutes.

Use only the provided priority scores and travel information.

Travel times are directional.
Do NOT assume that travel times are symmetric.

For example:
Ja-Ela -> Gampaha = 40 minutes
does NOT imply
Gampaha -> Ja-Ela = 40 minutes.

You may derive a travel time only when it can be calculated directly
by chaining the provided travel legs in the same direction.

If a required travel time cannot be derived directly from the given
directional information, return "Unknown".

Do not claim that a strategy maximizes lives saved or life-saving impact,
because the system only evaluates priority scores and known travel times.

Base the final decision only on priority scores, service order,
and supported travel-time information.
""",
   format="""
Branch 1 - Highest Score First
Service Order: [incident/location order]
Known Travel Time: [time or Unknown]
Reasoning: [brief evaluation]

Branch 2 - Closest First
Service Order: [incident/location order]
Known Travel Time: [time or Unknown]
Reasoning: [brief evaluation]

Branch 3 - Furthest First
Service Order: [incident/location order]
Known Travel Time: [time or Unknown]
Reasoning: [brief evaluation]

Comparison:
[compare all three strategies using only supported information]

Optimal Route:
[selected service order]

Reason:
[brief justification]
"""
    )

    response = llm.chat(
        [{"role": "user", "content": prompt_text}],
        temperature=0.2,
        task_type="reasoning"
    )

    return response["text"].strip()

In [9]:
model = pick_model(
    provider="groq",
    technique="tot_reasoning"
)

print(model)

llm = LLMClient(
    "groq",
    model
)

openai/gpt-oss-120b


In [10]:
scored_incidents_text = "\n".join(
    [
        f"Incident {item['id']} | "
        f"Area: {item['area']} | "
        f"Main Need: {item['main_need']} | "
        f"Priority Score: {item['score']}/10"
        for item in scored_incidents
    ]
)

print(scored_incidents_text)

Incident 1 | Area: Gampaha | Main Need: Water | Priority Score: 5/10
Incident 2 | Area: Ja-Ela | Main Need: Insulin | Priority Score: 8/10
Incident 3 | Area: Ragama | Main Need: Rescue | Priority Score: 8/10


In [11]:
tot_result = plan_rescue_strategy(
    scored_incidents_text,
    llm
)

print(tot_result)

**Branch 1 – Highest Score First**  
Service Order: Ragama → Ja‑Ela → Gampaha  
Known Travel Time: 0 min (Ragama→Ragama) + 10 min (Ragama→Ja‑Ela) + 40 min (Ja‑Ela→Gampaha) = **50 minutes**  
Reasoning: Both Ragama and Ja‑Ela have the highest priority (8). The tie‑breaker uses the explicit travel time from the start; Ragama is 0 min away, Ja‑Ela is 10 min away, so Ragama is served first. After that the only known leg to the remaining incident is Ja‑Ela → Gampaha (40 min).  

**Branch 2 – Closest First**  
Service Order: Ragama → Ja‑Ela → Gampaha  
Known Travel Time: 0 min (Ragama→Ragama) + 10 min (Ragama→Ja‑Ela) + 40 min (Ja‑Ela→Gampaha) = **50 minutes**  
Reasoning: Starting at Ragama, the nearest incident is the one at the same location (0 min). The next nearest, using only the provided travel times, is Ja‑Ela (10 min). The only remaining known leg is Ja‑Ela → Gampaha (40 min).  

**Branch 3 – Furthest First**  
Service Order: Gampaha → Ja‑Ela → Ragama  
Known Travel Time: Ragama → Ja

In [12]:
from pathlib import Path

output_path = Path("../output/logistics_commander.md")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:

    file.write("# Part 3 - Logistics Commander\n\n")

    # Part A
    file.write("## Part A - CoT Priority Scoring\n\n")

    for item in scored_incidents:

        file.write(f"### Incident {item['id']}\n\n")

        file.write(f"- Area: {item['area']}\n")
        file.write(f"- Main Need: {item['main_need']}\n")
        file.write(f"- Priority Score: {item['score']}/10\n\n")

        file.write("#### CoT Analysis\n\n")
        file.write(item["cot_result"])
        file.write("\n\n")

    # Part B
    file.write("## Part B - ToT Rescue Strategy\n\n")
    file.write(tot_result)
    file.write("\n")

print(f"Saved to: {output_path}")

Saved to: ..\output\logistics_commander.md
